# 08 — YOLO evaluation

Evaluates the saved checkpoint on the held-out `test` split, analyses errors at
IoU 0.5, compares validation and test scores, logs the evaluation run and
registers the best training run in the MLflow model registry.

## 1. Test metrics

Only the training hyperparameters are reused; the test split is never seen
during training.

In [ ]:
from IPython.display import Image, display

from diagnostic_des_tumeurs.config import Paths, YOLO_EXPERIMENT, YoloConfig
from diagnostic_des_tumeurs.yolo.functions.evaluation import evaluate
from diagnostic_des_tumeurs.yolo.functions.tracking import (
    log_evaluation,
    register_best,
    save_comparison,
)

paths = Paths.from_root()
config = YoloConfig()
evaluation = evaluate(paths, config)
run_id = log_evaluation(evaluation, paths, YOLO_EXPERIMENT)
print("MLflow run:", run_id)
evaluation.metrics

## 2. Per-class metrics and TP/FP/FN analysis

The error table uses a greedy match at IoU 0.5 with a confidence threshold of
0.25. It complements the Ultralytics mAP metrics with explicit counts.

In [ ]:
display(evaluation.per_class)
display(evaluation.error_table)

## 3. Validation vs test

A small gap means the model generalises; a large negative gap suggests
over-fitting to the validation split.

In [ ]:
display(evaluation.comparison)
print("Verdict:", evaluation.verdict)

## 4. Plots

Ground truth / prediction pairs plus the per-class validation-vs-test bars.

In [ ]:
display(Image(filename=str(evaluation.figure_paths["predictions"])))
if "validation_vs_test" in evaluation.figure_paths:
    display(Image(filename=str(evaluation.figure_paths["validation_vs_test"])))

## 5. Compare runs and register the best model

The registry uses the SQLite backend (`mlflow.db`) because a plain file store
does not support the model registry. The best training run is selected by
validation `mAP50-95` and aliased as `best`; `registered_model.json` stores the
version, run id and metric for later reuse.

In [ ]:
comparison = save_comparison(YOLO_EXPERIMENT, paths)
display(comparison)
registered = register_best(YOLO_EXPERIMENT, paths)
registered